In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import json
import os
from PIL import Image
from glob import glob
import numpy as np
from tqdm import tqdm
from torchvision import transforms
from pytorchvideo.models.hub import x3d_xs
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
# 디바이스 설정
device = torch.device("cuda:2" if torch.cuda.is_available() else "cpu")
print("✅ device:", device)

✅ device: cuda:2


In [ ]:

key_value="필요한 물품 준비"
key_list=["손소독제", "주사기", "투약카드", "tray", "알콜솜"]
# key_value="사용한 물품 정리"
# key_list=["주사바늘 되씌우지 않음","주사바늘 손상성 폐기물 버림"]
  
params = {
    "image_size": 224,
    "frame_size": 50,
    "num_classes": len(key_list),
    "dim": (64, 128, 256, 512),
    "depth": (3, 4, 8, 3),
    "batch_size": 8,
    "mhsa_types": ('l', 'l', 'g', 'g'),
    "epoch": 200,
    "data_path": '../../data/',
    "second": '10sec',
    "class_name": key_value,
    "label_path": "../../data/label/check_list/",
    "image_channel": 3
}
params["second"]=f'{params["frame_size"]//5}sec'

In [14]:

trans = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
])

# 1. 파일 리스트 생성
file_list = [f"D{str(i+1).zfill(3)}" for i in range(200)]
remove_items = ['D151', 'D159', 'D187', 'D080']
filtered_lst = [item for item in file_list if item not in remove_items]
split = int(len(filtered_lst) * 0.8)
filtered_lst = filtered_lst[split:]
# 2. 영상 데이터 및 라벨 저장 공간
train_images = torch.zeros(len(filtered_lst), 3, params['image_channel'], params['frame_size'],
                           params['image_size'], params['image_size'])  # [N, 3, C, T, H, W]
image_label = torch.zeros(len(filtered_lst),len(key_list))  # [N, num_classes]

# 3. 데이터 로딩
for i in tqdm(range(len(filtered_lst))):
    sample_id = filtered_lst[i]
    with open(params['label_path'] + sample_id + '.json', 'r') as f:
        check = json.load(f)

    base_path = params['data_path'] + params["second"] + '/' + params["class_name"] + '/' + sample_id
    image_list_1 = sorted(glob(base_path + '/1/*.png'))
    image_list_2 = [f.replace('/1/', '/2/') for f in image_list_1]
    image_list_3 = [f.replace('/1/', '/3/') for f in image_list_1]
    for k in range(len(key_list)):
        label = 1 if check['행동'][params["class_name"]][key_list[k]] else 0
        image_label[i,k]= label

    for j in range(params['frame_size']):
        for vid_idx, image_list in enumerate([image_list_1, image_list_2, image_list_3]):
            img = Image.open(image_list[j]).convert('RGB').resize((params['image_size'], params['image_size']))
            train_images[i, vid_idx, :, j] = trans(img)

# 4. CustomDataset 클래스 수정
class CustomDataset(Dataset):
    def __init__(self, args, video_tensor, labels, train=True):
        self.videos = video_tensor  # [N, 3, C, T, H, W]
        self.labels = labels
        self.args = args
        # 공간 증강: 랜덤 리사이즈 크롭, 랜덤 수평 뒤집기, 컬러 지터 등
        self.spatial_aug = transforms.Compose([
            transforms.RandomResizedCrop(args['image_size'], scale=(0.8,1.0)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        ])
        self.train = train
    def __getitem__(self, idx):
        video1 = self.videos[idx, 0]
        video2 = self.videos[idx, 1]
        video3 = self.videos[idx, 2]
        label = self.labels[idx]
        return video1, video2, video3, label

    def __len__(self):
        return len(self.videos)


test_dataset  = CustomDataset(params, train_images, image_label.float(), train=False)

test_dataloader  = DataLoader(test_dataset, batch_size=params['batch_size'], shuffle=False, drop_last=True)

100%|██████████| 40/40 [01:01<00:00,  1.53s/it]


In [15]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np
import pandas as pd

def test_multilabel_model(params, model_path, test_dataloader, key_list):
    model = Multix3d(num_classes=params['num_classes'])
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for video1, video2, video3, label in tqdm(test_dataloader, desc="🧪 Testing"):
            video1 = video1.to(device)
            video2 = video2.to(device)
            video3 = video3.to(device)
            label = label.to(device)

            output = model(video1, video2, video3)
            preds = torch.sigmoid(output).cpu().numpy() > 0.5
            labels = label.cpu().numpy()

            all_preds.append(preds)
            all_labels.append(labels)

    all_preds = np.concatenate(all_preds, axis=0).astype(int)
    all_labels = np.concatenate(all_labels, axis=0).astype(int)

    # 각 클래스별 지표 계산
    precision = precision_score(all_labels, all_preds, average=None, zero_division=0)
    recall = recall_score(all_labels, all_preds, average=None, zero_division=0)
    f1 = f1_score(all_labels, all_preds, average=None, zero_division=0)
    acc = (all_preds == all_labels).mean(axis=0)  # label 단위 accuracy

    class_names = [f"{params['class_name']} - {k}" for k in key_list]

    df_result = pd.DataFrame({
        "Class": class_names,
        "Accuracy": acc,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1
    })

    # 전체 평균
    macro_avg = df_result[["Accuracy", "Precision", "Recall", "F1-score"]].mean().to_dict()

    print("\n📊 Per-Class Metrics:")
    print(df_result)
    print("\n📈 Macro-Average Metrics:")
    for k, v in macro_avg.items():
        print(f"{k}: {v:.4f}")

    # 저장
    ca = params["class_name"]
    result_dir = f"../../result/action/"
    os.makedirs(result_dir, exist_ok=True)
    csv_path = os.path.join(result_dir, f"multilabel_{ca}.csv")
    df_result.to_csv(csv_path, index=False, encoding="utf-8-sig")
    print(f"\n✅ 결과 저장 완료: {csv_path}")

    # Confusion Matrix 시각화 저장
    cm_dir = os.path.join(result_dir, f"confusion_matrix")
    os.makedirs(cm_dir, exist_ok=True)

    for idx, class_name in enumerate(class_names):
        cm = confusion_matrix(all_labels[:, idx], all_preds[:, idx])
        plt.figure(figsize=(4, 3))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=["Not", "Yes"],
                    yticklabels=["Not", "Yes"])
        plt.xlabel("Predicted")
        plt.ylabel("Ground Truth")
        plt.title(f"Confusion Matrix")
        plt.tight_layout()
        cm_path = os.path.join(cm_dir, f"{params['class_name']}_{class_name}.png")
        plt.savefig(cm_path)
        plt.close()
        print(f"🖼️ Confusion Matrix saved: {cm_path}")

    return df_result, macro_avg

In [16]:
model_path = f"../../model/{params['class_name']}/best_model_{params['second']}.pt"
df_result, macro_avg = test_multilabel_model(params, model_path, test_dataloader, key_list)



🧪 Testing: 100%|██████████| 5/5 [00:02<00:00,  2.31it/s]



📊 Per-Class Metrics:
                         Class  Accuracy  Precision    Recall  F1-score
0     사용한 물품 정리 - 주사바늘 되씌우지 않음     0.600        0.5  0.500000  0.500000
1  사용한 물품 정리 - 주사바늘 손상성 폐기물 버림     0.675        1.0  0.458333  0.628571

📈 Macro-Average Metrics:
Accuracy: 0.6375
Precision: 0.7500
Recall: 0.4792
F1-score: 0.5643

✅ 결과 저장 완료: ../../result/action/multilabel_사용한 물품 정리.csv
🖼️ Confusion Matrix saved: ../../result/action/confusion_matrix/사용한 물품 정리_사용한 물품 정리 - 주사바늘 되씌우지 않음.png
🖼️ Confusion Matrix saved: ../../result/action/confusion_matrix/사용한 물품 정리_사용한 물품 정리 - 주사바늘 손상성 폐기물 버림.png
